# Part 4 -- Testing rank fusion on the failures

Part 3 ended with a build order: keyword ranking first, mixed into the
scoring. This part built the mix and measured it. Before the first run,
we fixed the rules of the test: what the mix must beat, and what counts
as a win. The mix lost. The result we ship is plain keyword ranking.

Code cells are excerpts from the real scripts; outputs are pasted from
the actual runs.

## 1. What must the mix beat?

Part 3 priced keyword ranking at 101 rescues -- and said its measured
number counted rescues only. A fix changes the ranking for every
question, so it must be scored in both directions: failures it fixes,
and successes it breaks. So the first step was the missing half: run
plain keyword ranking on the 95 successes and count the damage.

In [ ]:
breaks = [q for q in successes if keyword_depth(q) > 10]

plain keyword ranking on the 95 successes: breaks 12

system                              fixes   breaks   total of 266
embedder alone (today)                  -        -             95
plain keyword ranking alone           101       12            184


The bar is the bottom row. A mix that cannot beat plain keyword ranking
means: ship plain keyword ranking. We wrote that rule down, with the
verdict tiers and the tie-breakers, before computing any mix score --
so a weak result could not soften the test after the fact.

## 2. The mix

The mix is one combined ranking built from both signals. The recipe is
rank fusion (the standard form, reciprocal rank fusion): an article's
fused score adds one part per ranking, each part growing as the
article's rank improves. No dial to tune.

In [ ]:
fused_score = 1/(60 + keyword_rank) + 1/(60 + embedder_rank)
# 60 is the literature default; articles re-ranked by fused_score

check: the fused run reproduces both inputs exactly
  embedder ranking: all 266 depths match the recorded baseline
  keyword ranking:  101 fixes / 12 breaks reproduced


## 3. What came back

In [ ]:
verdict(mix)

fixes            64
breaks            6     (all 6 are among plain keywords' 12; new damage: 0)
total           153     of 266
the bar         184
verdict         the mix loses by 31


The mix is the safer system -- half the breaks -- and the weaker one.
Where the 31 went: the fusion dropped a large share of the keyword
wins, and bought few of its own.

In [ ]:
lost = [q for q in keyword_fixes if fused_depth(q) > 10]

of plain keywords' 101 fixes:
  the mix kept                   57
  the mix lost                   44
    landing at rank 11-20        33     (just past the top-10 cut)
    landing at rank 21-50        11
fixes the mix added that plain keywords lack: 7


The embedding is not noise on the lost questions -- its closest needed
set sits around rank 50 of 360, far better than chance. But rank 50 is
not top-10 signal, and the fusion weighs both rankings equally. On the
questions keywords win, averaging in a rank-50 opinion drags a third of
the wins just past the cut.

## 4. The weighted version

The fusion above has no dial. The weighted version has one: a slider
between all-keywords and all-embedding. We report its curve as numbers
only. Picking the best slider setting from these same 266 questions and
calling it the result would be grading the fix on its own answer key --
any adopted setting needs evidence from questions it was not tuned on.

In [ ]:
for alpha in 0.1 ... 0.9:     # weight on the keyword side
    score = alpha * z(keyword_score) + (1 - alpha) * z(embedder_score)

alpha    fixes   breaks   total
0.1         37        3     129
0.2         57        6     146
0.3         73        8     160
0.4         85       11     169
0.5         88       10     173
0.6         90       12     173
0.7         91       12     174
0.8         96       12     179
0.9         99       12     182


Every row is below the bar. Inside this family, on these questions, no
slider setting beat plain keywords.

## 5. What Part 4 settles

The build order's first step is done, with a twist: the winning build
is not the mix -- it is plain keyword ranking, adopted as the new
system to beat. Two smaller readings from the same run:

In [ ]:
scorecard(mix, causes)

the mix against Part 2's causes
  "nothing to grab" failures     fixed  2 of 34    (keywords alone: 7)
  near-copy crowding failures    fixed 27 of 99    (the mix overall: 64 of 171)


The failures that give keywords nothing to work with stayed out of
reach -- the embedding side of the mix did not open them. And crowding
stayed nobody's fix, as Part 3 measured for every candidate.

What is next: stacking the other measured fixes on top of plain
keywords (best-sentence scoring, dropping the unlisted copies), and
decomposition for the questions no ranking fix reaches.

**Score a fix against the strongest simple alternative, not against
the system it replaces.**